# Late Fusion Pipeline (Phase 2a)

**Goal**: Combine R-GCN graph scores with text embedding cosine similarity scores.

`s_final(d) = β · s_graph(d) + (1-β) · s_LLM(d)`

**Pipeline**:
1. Generate Tier 1 + Tier 2 descriptions from PrimeKG metadata
2. Encode Tier 2 with 4 encoders × 4 projections → select best combo by proxy MRR
3. (Optional) GPT-4o enrichment with best encoder
4. Full late fusion: beta sweep with 5-fold CV → evaluate on test diseases

## 0. Setup

In [1]:
!pip install -q transformers adapters torch torch-geometric wandb scikit-learn openai scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 152.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 122.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT = "/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702"
sys.path.insert(0, PROJECT)
%cd {PROJECT}

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/16u6rgAZFbJh8HQhxtjFa2_tq8qbi6dht/BMI702


In [4]:
import json
import logging
import numpy as np
import pandas as pd
import torch

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


---
## 1. Generate Descriptions (Tier 1 + Tier 2)

In [ ]:
!ls data/primekg/

edges.csv  kg.csv  nodes.csv


In [ ]:
# Generate Tier 2 descriptions (name + 1-hop KG context)
!python scripts/generate_descriptions.py \
    --data-dir data/primekg \
    --split-dir data/splits \
    --output-dir data/descriptions \
    --tier tier2

INFO: Loading PrimeKG...
/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/scripts/generate_descriptions.py:420: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  kg_df = pd.read_csv(data_dir / "kg.csv")
INFO: Loaded 129375 nodes, 8100498 edges
INFO: Split: 431 train, 108 test diseases
INFO: Phenotypes associated with eligible diseases: 3518
INFO: Building tier2 drug descriptions...
INFO: Built 7957 drug descriptions
INFO: Building tier2 phenotype descriptions...
INFO: Built 3518 phenotype descriptions
INFO: Saved: data/descriptions/drugs_tier2.json (7957 drugs)
INFO: Saved: data/descriptions/phenotypes_tier2.json (3518 phenotypes)
INFO: Sample drug description: Copper. Targets: A1BG, A2M, ACTG1, ACTN1, ACY1, AFM, AGT, AHCY, AHSG, AKR1A1.
INFO: Sample phenotype description: Growth abnormality. Associated genes: TET3, ZMIZ1, ZNF292. Associated conditions: Andersen-Tawil syndrome, CHIME syndrome, Czech dysplasia, metatarsal type,

In [ ]:
# Generate Tier 1 descriptions (name only, for ablation)
!python scripts/generate_descriptions.py \
    --data-dir data/primekg \
    --split-dir data/splits \
    --output-dir data/descriptions \
    --tier tier1

INFO: Loading PrimeKG...
['edges.csv', 'nodes.csv', 'kg.csv']
/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/scripts/generate_descriptions.py:422: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  kg_df = pd.read_csv(data_dir / "kg.csv")
INFO: Loaded 129375 nodes, 8100498 edges
INFO: Split: 431 train, 108 test diseases
INFO: Phenotypes associated with eligible diseases: 3518
INFO: Building tier1 drug descriptions...
INFO: Built 7957 drug descriptions
INFO: Building tier1 phenotype descriptions...
INFO: Built 3518 phenotype descriptions
INFO: Saved: data/descriptions/drugs_tier1.json (7957 drugs)
INFO: Saved: data/descriptions/phenotypes_tier1.json (3518 phenotypes)
INFO: Sample drug description: Copper
INFO: Sample phenotype description: Growth abnormality


In [ ]:
# Verify
for tier in ["tier1", "tier2"]:
    for entity in ["drugs", "phenotypes"]:
        path = f"data/descriptions/{entity}_{tier}.json"
        with open(path) as f:
            data = json.load(f)
        sample = next(iter(data.values()))
        print(f"{entity}_{tier}: {len(data)} entries")
        print(f"  Sample: {sample['text'][:120]}...\n")

drugs_tier1: 7957 entries
  Sample: Copper...

phenotypes_tier1: 3518 entries
  Sample: Growth abnormality...

drugs_tier2: 7957 entries
  Sample: Copper. Targets: A1BG, A2M, ACTG1, ACTN1, ACY1, AFM, AGT, AHCY, AHSG, AKR1A1....

phenotypes_tier2: 3518 entries
  Sample: Growth abnormality. Associated genes: TET3, ZMIZ1, ZNF292. Associated conditions: Andersen-Tawil syndrome, CHIME syndrom...



---
## 2. Encoder + Projection Selection

Encode Tier 2 descriptions with all 4 encoders × 4 projections, then pick the best combo by proxy MRR on training diseases.

In [ ]:
# Encode all 16 combos (4 encoders × 4 projections)
# This takes ~20-40 min total. To save time, start with just pca + none.

encoders = ["pubmedbert", "biolinkbert", "biomedbert", "specter2"]
#encoders = ["specter2"]
#encoders = ["biomedbert"]
projections = ["pca", "none", "linear", "nonlinear_ae"]

for enc in encoders:
    for proj in projections:
        print(f"\n{'='*60}")
        print(f"Encoding: {enc} / {proj}")
        print(f"{'='*60}")
        !python scripts/cache_embeddings.py \
            --desc-dir data/descriptions \
            --output-dir data/embeddings \
            --encoder {enc} \
            --tier tier2 \
            --projection {proj} \
            --device cuda

In [ ]:
# Compute proxy MRR for each combo
from src.data.primekg_loader import load_primekg, build_supervision_maps
from src.data.disease_split import load_split
from src.evaluation.late_fusion_eval import load_llm_scores
from src.evaluation.metrics import reciprocal_rank

nodes_df, edges_df, kg_df = load_primekg("data/primekg")
train_diseases, test_diseases, train_pairs, test_pairs = load_split("data/splits")
supervision = build_supervision_maps(
    kg_df, nodes_df, train_diseases, test_diseases, train_pairs, test_pairs
)
disease_to_phenotypes = supervision["disease_to_phenotypes"]
train_disease_to_drugs = supervision["train_disease_to_drugs"]
drug_indices_arr = np.array(sorted(supervision["drug_indices"]))

print(f"Train diseases: {len(train_diseases)}")
print(f"Test diseases: {len(test_diseases)}")
print(f"Drug nodes: {len(drug_indices_arr)}")

/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/src/data/primekg_loader.py:45: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  kg = pd.read_csv(data_dir / "kg.csv")


Train diseases: 431
Test diseases: 108
Drug nodes: 7957


In [ ]:
selection_results = []

for enc in encoders:
    for proj in projections:
        drug_path = f"data/embeddings/{enc}/tier2/{proj}/drug_embeddings.pt"
        pheno_path = f"data/embeddings/{enc}/tier2/{proj}/phenotype_embeddings.pt"

        try:
            llm_scores = load_llm_scores(
                train_diseases, disease_to_phenotypes, drug_path, pheno_path
            )
        except FileNotFoundError:
            print(f"  {enc}/{proj}: SKIPPED (not cached)")
            continue

        mrrs = []
        for d_idx, scores in llm_scores.items():
            true_drugs = list(train_disease_to_drugs.get(d_idx, []))
            if not true_drugs:
                continue
            ranked = drug_indices_arr[np.argsort(-scores)].tolist()
            mrrs.append(reciprocal_rank(ranked, true_drugs))

        proxy_mrr = np.mean(mrrs) if mrrs else 0.0
        selection_results.append({"encoder": enc, "projection": proj, "proxy_MRR": proxy_mrr})
        print(f"  {enc}/{proj}: proxy MRR = {proxy_mrr:.4f} ({len(mrrs)} diseases)")

selection_df = pd.DataFrame(selection_results).sort_values("proxy_MRR", ascending=False)
print("\n=== Encoder + Projection Ranking ===")
print(selection_df.to_string(index=False))

  pubmedbert/pca: proxy MRR = 0.1948 (431 diseases)
  pubmedbert/none: proxy MRR = 0.2264 (431 diseases)
  pubmedbert/linear: proxy MRR = 0.1825 (431 diseases)
  pubmedbert/nonlinear_ae: proxy MRR = 0.2303 (431 diseases)
  biolinkbert/pca: proxy MRR = 0.1965 (431 diseases)
  biolinkbert/none: proxy MRR = 0.2083 (431 diseases)
  biolinkbert/linear: proxy MRR = 0.2038 (431 diseases)
  biolinkbert/nonlinear_ae: proxy MRR = 0.2622 (431 diseases)
  biomedbert/pca: proxy MRR = 0.1955 (431 diseases)
  biomedbert/none: proxy MRR = 0.2022 (431 diseases)
  biomedbert/linear: proxy MRR = 0.1895 (431 diseases)
  biomedbert/nonlinear_ae: proxy MRR = 0.1700 (431 diseases)
  specter2/pca: proxy MRR = 0.2153 (431 diseases)
  specter2/none: proxy MRR = 0.2138 (431 diseases)
  specter2/linear: proxy MRR = 0.2085 (431 diseases)
  specter2/nonlinear_ae: proxy MRR = 0.2135 (431 diseases)

=== Encoder + Projection Ranking ===
    encoder   projection  proxy_MRR
biolinkbert nonlinear_ae   0.262170
 pubmedber

In [ ]:
# Select the best combo
best_row = selection_df.iloc[0]
BEST_ENCODER = best_row["encoder"]
BEST_PROJECTION = best_row["projection"]

print(f"*** Best encoder:    {BEST_ENCODER}")
print(f"*** Best projection: {BEST_PROJECTION}")
print(f"*** Proxy MRR:       {best_row['proxy_MRR']:.4f}")

*** Best encoder:    biolinkbert
*** Best projection: nonlinear_ae
*** Proxy MRR:       0.2622


---
## 3. Tier 1 vs Tier 2 Ablation (Optional)

Compare proxy MRR of Tier 1 (name only) vs Tier 2 (name + KG context) with the best encoder.

In [ ]:
# Encode Tier 1 with the best combo
!python scripts/cache_embeddings.py \
    --desc-dir data/descriptions \
    --output-dir data/embeddings \
    --encoder {BEST_ENCODER} \
    --tier tier1 \
    --projection {BEST_PROJECTION} \
    --device cuda

INFO: Encoder: biolinkbert (michiyasunaga/BioLinkBERT-base)
INFO: Pooling: mean
INFO: Tier: tier1, Projection: nonlinear_ae
INFO: Seed: 42
INFO: Loading drugs descriptions from data/descriptions/drugs_tier1.json
INFO: Loaded 7957 drugs descriptions
INFO: Encoding drugs with biolinkbert...
INFO: NumExpr defaulting to 8 threads.
tokenizer_config.json: 100% 379/379 [00:00<00:00, 2.85MB/s]
vocab.txt: 225kB [00:00, 61.5MB/s]
tokenizer.json: 447kB [00:00, 93.6MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 867kB/s]
config.json: 100% 559/559 [00:00<00:00, 5.04MB/s]
2026-04-22 07:43:34.466671: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776843814.487252    7895 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776843814.493403    7895 cu

In [ ]:
# Tier 1 proxy MRR
drug_path_t1 = f"data/embeddings/{BEST_ENCODER}/tier1/{BEST_PROJECTION}/drug_embeddings.pt"
pheno_path_t1 = f"data/embeddings/{BEST_ENCODER}/tier1/{BEST_PROJECTION}/phenotype_embeddings.pt"

llm_scores_t1 = load_llm_scores(
    train_diseases, disease_to_phenotypes, drug_path_t1, pheno_path_t1
)

mrrs_t1 = []
for d_idx, scores in llm_scores_t1.items():
    true_drugs = list(train_disease_to_drugs.get(d_idx, []))
    if not true_drugs:
        continue
    ranked = drug_indices_arr[np.argsort(-scores)].tolist()
    mrrs_t1.append(reciprocal_rank(ranked, true_drugs))

tier1_mrr = np.mean(mrrs_t1)
tier2_mrr = best_row["proxy_MRR"]

print(f"Tier 1 (name only)    proxy MRR: {tier1_mrr:.4f}")
print(f"Tier 2 (name + KG)    proxy MRR: {tier2_mrr:.4f}")
print(f"Delta (Tier2 - Tier1):           {tier2_mrr - tier1_mrr:+.4f}")

Tier 1 (name only)    proxy MRR: 0.0160
Tier 2 (name + KG)    proxy MRR: 0.2622
Delta (Tier2 - Tier1):           +0.2462


---
## 4. GPT-4o Description Enrichment (Optional)

Enrich Tier 2 descriptions with GPT-4o. **Cost: ~$8.** Skip if you want to test fusion with Tier 2 first.

In [ ]:
import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))


Enter OPENAI_API_KEY: ··········
OPENAI_API_KEY set: True


In [ ]:
# Quick dry-run on ~10 samples before full GPT-4o enrichment
import json
import random
from openai import OpenAI
from scripts.generate_descriptions import DRUG_PROMPT, PHENOTYPE_PROMPT

client = OpenAI()
rng = random.Random(42)

with open("data/descriptions/drugs_tier2.json") as f:
    drug_desc = json.load(f)
with open("data/descriptions/phenotypes_tier2.json") as f:
    pheno_desc = json.load(f)

sample_drugs = rng.sample(list(drug_desc.items()), k=min(5, len(drug_desc)))
sample_phenos = rng.sample(list(pheno_desc.items()), k=min(5, len(pheno_desc)))

def _preview(item, entity_type):
    idx, entry = item
    prompt = (DRUG_PROMPT if entity_type == "drug" else PHENOTYPE_PROMPT).format(
        tier2_text=entry["text"]
    )
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=120,
    )
    text = resp.choices[0].message.content.strip().replace("\n", " ")
    print(f"[{entity_type}:{idx}] {entry['name']} -> {text[:180]}...")

for item in sample_drugs:
    _preview(item, "drug")
for item in sample_phenos:
    _preview(item, "phenotype")

print("Sample check complete. If outputs look good, set RUN_FULL_GPT4O=True in the next cell.")


[drug:19250] Pentostatin -> Pentostatin is an antineoplastic agent that functions as an adenosine deaminase (ADA) inhibitor, leading to the accumulation of deoxyadenosine triphosphate, which is toxic to lymph...
[drug:14924] Diazoxide -> Diazoxide is a potassium channel opener that primarily acts by activating K_ATP channels, particularly targeting the KCNJ11 subunit, leading to hyperpolarization of pancreatic beta...
[drug:14216] Warfarin -> Warfarin is an anticoagulant that functions by inhibiting the vitamin K epoxide reductase complex 1 (VKORC1), leading to a reduction in the synthesis of active clotting factors II,...
[drug:20086] Alpha-Methylisocitric Acid -> Alpha-Methylisocitric Acid is a metabolic compound that targets aconitase 2 (ACO2), an enzyme involved in the citric acid cycle. By inhibiting ACO2, it disrupts the conversion of c...
[drug:16265] Soybean oil -> Soybean oil is a natural lipid that primarily targets the peroxisome proliferator-activated receptor alpha (PPARA)

In [ ]:
# (Optional) Faster GPT-4o enrichment: concurrent requests + 429 backoff
# This is an alternative to running scripts/generate_descriptions.py --use-llm (which is serial).

import asyncio
import json
import os
import random
import time

from openai import AsyncOpenAI

try:
    from openai import RateLimitError, APIError, APITimeoutError, APIConnectionError
except Exception:  # older SDKs
    RateLimitError = APIError = APITimeoutError = APIConnectionError = Exception

from scripts.generate_descriptions import DRUG_PROMPT, PHENOTYPE_PROMPT

async_client = AsyncOpenAI()


async def _call_with_backoff(*, prompt: str, model: str, max_retries: int = 8, base_delay_s: float = 0.5, max_delay_s: float = 30.0):
    """Call the API with exponential backoff (+ jitter) on transient failures (429/5xx/timeouts)."""
    for attempt in range(max_retries):
        try:
            resp = await async_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=200,
            )
            return resp.choices[0].message.content.strip()
        except (RateLimitError, APITimeoutError, APIConnectionError, APIError):
            if attempt >= max_retries - 1:
                raise
            sleep_s = min(max_delay_s, base_delay_s * (2 ** attempt))
            sleep_s *= 0.5 + random.random()  # jitter in [0.5, 1.5)
            await asyncio.sleep(sleep_s)


async def enrich_json_concurrent(
    *,
    input_path: str,
    output_path: str,
    entity_type: str,
    model: str = "gpt-4o",
    max_in_flight: int = 20,
    checkpoint_every: int = 50,
):
    """Concurrent, resumable enrichment that writes the same JSON format as scripts/generate_descriptions.py."""
    assert entity_type in {"drug", "phenotype"}
    prompt_template = DRUG_PROMPT if entity_type == "drug" else PHENOTYPE_PROMPT

    with open(input_path) as f:
        base = json.load(f)  # {str(node_idx): {name,text}}

    enriched = {}
    if os.path.exists(output_path):
        with open(output_path) as f:
            enriched = json.load(f)
        print(f"Resuming {entity_type}: {len(enriched)}/{len(base)} already enriched")

    q: asyncio.Queue = asyncio.Queue()
    for node_idx, item in base.items():
        if node_idx not in enriched:
            q.put_nowait((node_idx, item))

    progress = {"done": 0}
    t0 = time.time()

    async def worker(worker_id: int):
        while True:
            try:
                node_idx, item = q.get_nowait()
            except asyncio.QueueEmpty:
                return

            prompt = prompt_template.format(tier2_text=item["text"])
            try:
                text = await _call_with_backoff(prompt=prompt, model=model)
                enriched[node_idx] = {"name": item.get("name", ""), "text": text}
            except Exception as e:
                # Fallback: keep Tier 2 text so downstream sections can still run
                enriched[node_idx] = {"name": item.get("name", ""), "text": item.get("text", "")}
                enriched[node_idx]["_error"] = str(e)

            progress["done"] += 1
            if progress["done"] % checkpoint_every == 0:
                with open(output_path, "w") as f:
                    json.dump(enriched, f, indent=2)
                elapsed = time.time() - t0
                rate = progress["done"] / max(elapsed, 1e-6)
                print(f"Checkpoint {entity_type}: +{progress['done']} (rate={rate:.2f}/s), saved -> {output_path}")

            q.task_done()

    workers = [asyncio.create_task(worker(i)) for i in range(max_in_flight)]
    await asyncio.gather(*workers)

    with open(output_path, "w") as f:
        json.dump(enriched, f, indent=2)
    print(f"Done {entity_type}: saved {len(enriched)}/{len(base)} -> {output_path}")
    return enriched





In [ ]:
# Progress check: how many GPT-4o descriptions have been written to disk so far?
# Safe to run while cell-21 is still in progress — enrich_with_llm checkpoints
# to disk every 50 entities.
from pathlib import Path

def _progress(partial_path: str, total_path: str, label: str):
    partial = Path(partial_path)
    with open(total_path) as f:
        total = len(json.load(f))
    if not partial.exists():
        print(f"{label:12s}  0 / {total}  (0.0%)  — no partial file yet")
        return
    with open(partial) as f:
        done = json.load(f)
    n = len(done)
    pct = 100.0 * n / total if total else 0.0
    remaining = total - n
    print(f"{label:12s}  {n} / {total}  ({pct:5.1f}%)  remaining: {remaining}")
    sample = next(iter(done.values()))
    snippet = sample["text"].replace("\n", " ")[:140]
    print(f"             sample: {sample['name']} -> {snippet}...")

_progress("data/descriptions/drugs_gpt4o.json",      "data/descriptions/drugs_tier2.json",      "drugs")
_progress("data/descriptions/phenotypes_gpt4o.json", "data/descriptions/phenotypes_tier2.json", "phenotypes")


In [ ]:
# Example usage (run one or both):
await enrich_json_concurrent(
    input_path="data/descriptions/drugs_tier2.json",
    output_path="data/descriptions/drugs_gpt4o.json",
    entity_type="drug",
    model="gpt-4o",
    max_in_flight=20,
)
await enrich_json_concurrent(
    input_path="data/descriptions/phenotypes_tier2.json",
    output_path="data/descriptions/phenotypes_gpt4o.json",
    entity_type="phenotype",
    model="gpt-4o",
    max_in_flight=20,
)

Resuming drug: 2000/7957 already enriched
Checkpoint drug: +50 (rate=7.74/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +100 (rate=8.45/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +150 (rate=8.92/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +200 (rate=5.70/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +250 (rate=4.17/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +300 (rate=3.76/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +350 (rate=3.25/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +400 (rate=2.89/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +450 (rate=2.79/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +500 (rate=2.61/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +550 (rate=2.51/s), saved -> data/descriptions/drugs_gpt4o.json
Checkpoint drug: +600 (rate=2.43/s), saved -> data/descriptions/drug

{'22128': {'name': 'Functional abnormality of the bladder',
  'text': 'Functional abnormality of the bladder. Associated conditions: Alstrom syndrome, Ehlers-Danlos syndrome, musculocontractural, X-linked immunoneurologic disorder, autoimmune enteropathy and endocrinopathy - susceptibility to chronic infections syndrome, complex spastic paraplegia, detrusor sphincter dyssynergia (disease), distal 10q deletion syndrome, hereditary spastic paraplegia, interstitial cystitis, nephrogenic diabetes insipidus.',
  '_error': "Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}"},
 '22156': {'name': 'Abnormal renal morphology',
  'text': 'Abnormal renal morphology. Associated conditions: 15q overgrowth syndrome, 3p- syndrome, 48,XYYY syndro

In [ ]:
# Generate GPT-4o enriched descriptions (resumable if interrupted)
RUN_FULL_GPT4O = True  # set True after the sample check above
if RUN_FULL_GPT4O:
    !python scripts/generate_descriptions.py \
        --data-dir data/primekg \
        --split-dir data/splits \
        --output-dir data/descriptions \
        --tier tier2 \
        --use-llm \
        --llm-model gpt-4o
else:
    print("Skipped full GPT-4o enrichment. Set RUN_FULL_GPT4O=True to run.")


INFO: Loading PrimeKG...
['edges.csv', 'nodes.csv', 'kg.csv']
/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/scripts/generate_descriptions.py:422: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  kg_df = pd.read_csv(data_dir / "kg.csv")
INFO: Loaded 129375 nodes, 8100498 edges
INFO: Split: 431 train, 108 test diseases
INFO: Phenotypes associated with eligible diseases: 3518
INFO: Building tier2 drug descriptions...
INFO: Built 7957 drug descriptions
INFO: Building tier2 phenotype descriptions...
INFO: Built 3518 phenotype descriptions
INFO: Enriching drugs with gpt-4o...
INFO: Enriching 7957 drugs with gpt-4o
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/comple

In [ ]:
# Verify GPT-4o descriptions
with open("data/descriptions/drugs_gpt4o.json") as f:
    gpt4o_drugs = json.load(f)
print(f"GPT-4o drugs: {len(gpt4o_drugs)}")
sample = next(iter(gpt4o_drugs.values()))
print(f"Sample: {sample['text'][:200]}")

GPT-4o drugs: 7957
Sample: Copper is an essential trace element that plays a critical role in various physiological processes, including acting as a cofactor for enzymes involved in redox reactions, iron metabolism, and connect


In [ ]:
# Encode GPT-4o descriptions with best encoder + projection
!python scripts/cache_embeddings.py \
    --desc-dir data/descriptions \
    --output-dir data/embeddings \
    --encoder {BEST_ENCODER} \
    --tier gpt4o \
    --projection {BEST_PROJECTION} \
    --device cuda

usage: cache_embeddings.py [-h] [--desc-dir DESC_DIR]
                           [--output-dir OUTPUT_DIR] --encoder
                           {pubmedbert,biomedbert,biolinkbert,specter2}
                           [--tier {tier1,tier2,gpt4o}]
                           [--projection {pca,linear,nonlinear_ae,none}]
                           [--target-dim TARGET_DIM] [--batch-size BATCH_SIZE]
                           [--max-length MAX_LENGTH] [--device DEVICE]
                           [--seed SEED]
cache_embeddings.py: error: argument --encoder: invalid choice: '{BEST_ENCODER}' (choose from pubmedbert, biomedbert, biolinkbert, specter2)


In [ ]:
# GPT-4o proxy MRR
drug_path_gpt4o = f"data/embeddings/{BEST_ENCODER}/gpt4o/{BEST_PROJECTION}/drug_embeddings.pt"
pheno_path_gpt4o = f"data/embeddings/{BEST_ENCODER}/gpt4o/{BEST_PROJECTION}/phenotype_embeddings.pt"

llm_scores_gpt4o = load_llm_scores(
    train_diseases, disease_to_phenotypes, drug_path_gpt4o, pheno_path_gpt4o
)

mrrs_gpt4o = []
for d_idx, scores in llm_scores_gpt4o.items():
    true_drugs = list(train_disease_to_drugs.get(d_idx, []))
    if not true_drugs:
        continue
    ranked = drug_indices_arr[np.argsort(-scores)].tolist()
    mrrs_gpt4o.append(reciprocal_rank(ranked, true_drugs))

gpt4o_mrr = np.mean(mrrs_gpt4o)
print(f"Tier 1 (name only)    proxy MRR: {tier1_mrr:.4f}")
print(f"Tier 2 (name + KG)    proxy MRR: {tier2_mrr:.4f}")
print(f"GPT-4o enriched       proxy MRR: {gpt4o_mrr:.4f}")
print(f"Delta (GPT-4o - Tier2):          {gpt4o_mrr - tier2_mrr:+.4f}")

NameError: name 'BEST_ENCODER' is not defined

---
## 5. Full Late Fusion Experiment

Load trained R-GCN checkpoint, compute graph + LLM scores, calibrate beta with 5-fold CV, evaluate on 108 test diseases.

In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURE THESE BEFORE RUNNING
# ============================================================
CHECKPOINT_PATH = "data/weights/rgcn_best_model.pt"
DESC_TIER = "tier2"  # change to "gpt4o" after running Section 4
assert Path(CHECKPOINT_PATH).exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
# ============================================================


In [ ]:
import wandb
wandb.login()

In [ ]:
from src.evaluation.late_fusion_eval import run_late_fusion_experiment

config = {
    "data_dir": "data/primekg",
    "split_dir": "data/splits",
    "checkpoint_path": CHECKPOINT_PATH,
    "encoder_name": BEST_ENCODER,
    "desc_tier": DESC_TIER,
    "projection": BEST_PROJECTION,
    "embed_dir": "data/embeddings",
    "beta_search": [i / 10 for i in range(11)],
    "normalize": "minmax",
    "beta_cv_folds": 5,
    # R-GCN architecture (must match the trained checkpoint)
    "hidden_dim": 128,
    "num_bases": 10,
    "num_layers": 2,
    "num_heads": 4,
    "dropout": 0.2,
    "device": DEVICE.type,
    "results_dir": "results/tables",
}

results = run_late_fusion_experiment(config)

---
## 6. Results

In [ ]:
print(f"Best beta: {results['best_beta']}")
print(f"CV MRR:    {results['best_cv_mrr']:.4f}")
print()

print("=== Test Metrics (best fusion) ===")
for k, v in results["test_metrics"].items():
    print(f"  {k}: {v:.4f}")

print(f"\n=== Baselines ===")
print(f"  Pure graph  (beta=1.0): MRR={results['pure_graph_metrics']['MRR']:.4f}")
print(f"  Pure LLM    (beta=0.0): MRR={results['pure_llm_metrics']['MRR']:.4f}")

In [ ]:
# Beta sweep visualization
import matplotlib.pyplot as plt

betas = sorted(results["beta_to_mrr"].keys())
mrrs = [results["beta_to_mrr"][b] for b in betas]

plt.figure(figsize=(8, 5))
plt.plot(betas, mrrs, "o-", linewidth=2, markersize=8)
plt.axvline(results["best_beta"], color="red", linestyle="--", label=f"best β={results['best_beta']}")
plt.xlabel("β (graph weight)", fontsize=13)
plt.ylabel("CV MRR", fontsize=13)
plt.title(f"Late Fusion Beta Sweep ({BEST_ENCODER} / {DESC_TIER} / {BEST_PROJECTION})", fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"results/beta_sweep_{BEST_ENCODER}_{DESC_TIER}.png", dpi=150)
plt.show()

In [ ]:
# Cold-start stratification
tm = results["test_metrics"]
if "MRR_seen_all" in tm:
    strata_data = {
        "Stratum": ["All seen", "Some seen", "None seen (cold-start)"],
        "MRR": [tm["MRR_seen_all"], tm["MRR_seen_some"], tm["MRR_seen_none"]],
    }
    strata_df = pd.DataFrame(strata_data)
    print("=== Cold-Start Drug Stratification ===")
    print(strata_df.to_string(index=False))

In [ ]:
# Per-disease results
per_disease_df = results["per_disease_df"]

print(f"Per-disease results: {len(per_disease_df)} diseases")
print(f"\nWorst 5 (by MRR):")
print(per_disease_df.sort_values("MRR").head(5).to_string(index=False))

print(f"\nBest 5 (by MRR):")
print(per_disease_df.sort_values("MRR", ascending=False).head(5).to_string(index=False))

---
## 7. Compare Tier 2 vs GPT-4o (run after Section 4)

Run this section after completing GPT-4o enrichment to compare description tiers in full fusion.

In [ ]:
# Re-run with GPT-4o descriptions
config_gpt4o = config.copy()
config_gpt4o["desc_tier"] = "gpt4o"

results_gpt4o = run_late_fusion_experiment(config_gpt4o)

In [ ]:
# Side-by-side comparison
# AUROC / AUPRC = full-library per-disease, macro-averaged
# AUROC_balanced_micro / AUPRC_balanced_micro = TxGNN-style pooled 1:1 balanced
metric_keys = [
    "MRR", "R@1", "R@5", "R@10", "R@50",
    "AUROC", "AUPRC",
    "AUROC_balanced_micro", "AUPRC_balanced_micro",
]
comparison = pd.DataFrame({
    "Metric": metric_keys,
    "Pure Graph (β=1)": [results["pure_graph_metrics"].get(m, 0) for m in metric_keys],
    f"Tier 2 (β={results['best_beta']})": [results["test_metrics"].get(m, 0) for m in metric_keys],
    f"GPT-4o (β={results_gpt4o['best_beta']})": [results_gpt4o["test_metrics"].get(m, 0) for m in metric_keys],
})

print("=== Late Fusion: Description Tier Comparison ===")
print(comparison.to_string(index=False, float_format="%.4f"))